# Lab 7 — Data Preprocessing for AI: Preparing Data Without Cheating
**ARTI 303 — Programming for AI**

**Name:** _[your name]_  **Student ID:** _[your ID]_  **Section:** _[your section]_

---

### What this lab is really about

A model can only work with numbers, in a consistent shape, and it has
to be judged on data it has never seen. Getting your data into that
state is called **preprocessing**.

The danger is that preprocessing is easy to get subtly wrong in a way
that makes your results look better than they really are. Nothing
crashes and nothing warns you. This lab teaches the tools **and** the
one habit that keeps you honest.

This lab has **five ideas**, each short, each followed by an exercise
that checks itself.

| Part | Idea | You will be able to |
|---|---|---|
| 1 | Split first | divide data into train and test, and explain why this comes first |
| 2 | Encoding | turn categories into numbers without inventing an order |
| 3 | Scaling and missing values | put numeric columns on a common scale, using training data only |
| 4 | Pipelines | bundle every step into one object that can't be misused |
| 5 | Detect leakage | spot a leak in code or data, and validate your own output |

After Part 5 there is a short AI checkpoint and a list of resources.

---

## Setup

Same course folder, same environment as before:

```
arti303_lab/
├── .venv/
├── requirements.txt
├── lab01.ipynb ... lab06.ipynb
├── lab07.ipynb        <- this lab
├── data/
└── ...
```

**1.** Open `arti303_lab` in VS Code (`File → Open Folder`) — the folder
itself, not a lab file inside it.

**2.** Get the environment ready, in this order:

- Look for a `.venv` folder in the file list. If it isn't there, create
  it once: `python -m venv .venv`
- Activate it:

```bash
source .venv/bin/activate          # macOS / Linux / Git Bash
.venv\Scripts\Activate.ps1         # Windows PowerShell
```

- Check your prompt starts with `(.venv)`. If it shows `(base)`, you're
  in Anaconda's environment and packages will install to the wrong place.

**3.** Install today's library and record it:

```bash
python -m pip install scikit-learn
python -m pip freeze > requirements.txt
```

**4.** Point this notebook at the same environment:
`Cmd+Shift+P` (macOS) / `Ctrl+Shift+P` (Windows) → **Python: Select
Interpreter** → the entry showing `.venv`. Then restart the kernel.

### Check your setup

In [ ]:
import os
import sys

problems = []

if sys.prefix != sys.base_prefix:
    print("[ok]   Running inside a virtual environment")
else:
    print("[!]    NOT in a virtual environment")
    problems.append("Select the .venv interpreter, then restart the kernel")
print("       using:", sys.executable)
print()

for import_name, pip_name in [("pandas", "pandas"), ("numpy", "numpy"), ("sklearn", "scikit-learn")]:
    try:
        module = __import__(import_name)
        print(f"[ok]   {pip_name} {module.__version__}")
    except ImportError:
        print(f"[!]    {pip_name} is not installed in this environment")
        problems.append(f"Run: python -m pip install {pip_name}")

if os.path.exists("requirements.txt"):
    lines = [l.lower() for l in open("requirements.txt").read().splitlines() if l.strip()]
    if any("scikit-learn" in l for l in lines):
        print("[ok]   scikit-learn is recorded in requirements.txt")
    else:
        print("[!]    scikit-learn is missing from requirements.txt")
        problems.append("Run: python -m pip freeze > requirements.txt")
    if len(lines) > 100:
        print("[!]    requirements.txt has", len(lines), "packages - that looks like Anaconda's base")
        problems.append("Delete it, activate .venv, freeze again")
else:
    print("[!]    No requirements.txt found")
    problems.append("Run: python -m pip freeze > requirements.txt")

print()
print("Fix these first:" if problems else "Setup looks good.")
for item in problems:
    print("  -", item)

---
### The data

Today's dataset describes 200 students. The goal (not something we
build today, just the reason we prepare the data) is to flag students
who are **at risk**: those whose final GPA ends up below 2.5.

It is **synthetic**: the code below generates it with a fixed seed, so
everyone gets the same file. It is not real IAU data, but the departments
are real. A few values are missing, as in real data.

Run both cells.

In [ ]:
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)

# Generate the dataset (you don't need to understand this cell)
rng = np.random.default_rng(307)
n = 200
dept = rng.choice(["CE", "CS", "BME"], size=n, p=[0.40, 0.37, 0.23])
study_mode = rng.choice(["in-person", "online"], size=n, p=[0.70, 0.30])
study_hours = np.clip(rng.normal(14, 5, n), 2, 38).round(1)
attendance = np.clip(68 + 1.1 * study_hours + rng.normal(0, 6, n), 40, 100).round(0)
phone_hours = np.clip(rng.normal(4.5, 1.6, n) - 0.05 * (study_hours - 14), 0.5, 10).round(1)
credit_load = rng.integers(12, 19, n)
gpa = np.clip(
    2.0 + 0.075 * study_hours + 0.012 * (attendance - 80) - 0.05 * (phone_hours - 4.5)
    + rng.normal(0, 0.35, n),
    1.0, 5.0,
).round(2)

table = pd.DataFrame({
    "student_id": np.arange(3001, 3001 + n),
    "dept": dept,
    "study_mode": study_mode,
    "study_hours": study_hours,
    "attendance": attendance,
    "phone_hours": phone_hours,
    "credit_load": credit_load,
    "gpa": gpa,
    "at_risk": (gpa < 2.5).astype(int),
})
table.loc[rng.random(n) < 0.08, "phone_hours"] = np.nan
table.loc[rng.random(n) < 0.05, "attendance"] = np.nan
table.to_csv("data/students_prep.csv", index=False)

df = pd.read_csv("data/students_prep.csv")
print("shape:", df.shape)
df.head()

In [ ]:
print("missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print()
print("share of at-risk students:", round(df["at_risk"].mean(), 2))

---
# Part 1 · Split first

Everything a model learns must come from the **training set**. The
**test set** is its honest exam: data it has never seen.

"Never seen" applies to your **preprocessing** too. If a scaler
calculates an average using every row, including the test rows, then
information about the test set has already leaked into the training
data. The exam paper was visible while studying.

> **The rule: split first. Then learn anything from the training set
> only, and apply it to the test set.**

`train_test_split` does the split. Three arguments matter:

| Argument | What it does |
|---|---|
| `test_size` | the share held back for testing |
| `random_state` | makes the split reproducible, so everyone gets the same one |
| `stratify` | keeps the same share of each class in train and test. It matters here because only about one student in five is at risk. |

First, separate the **features** (what we know) from the **target**
(what we want to predict). We leave out `student_id` (just a label) and
`gpa`. Part 5 explains why `gpa` has to go.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["student_id", "gpa", "at_risk"])
y = df["at_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("train:", X_train.shape, "| test:", X_test.shape)
print("share at risk in train:", round(y_train.mean(), 3))
print("share at risk in test: ", round(y_test.mean(), 3))

Both shares are nearly identical, because of `stratify=y`. Without it,
a small test set could by chance contain almost no at-risk students.

### Exercise 1.1
Split `X` and `y` again, with these settings, into `X_tr, X_te, y_tr,
y_te`: **25%** for testing, `random_state=7`, stratified on `y`.

In [ ]:
# TODO: split X and y: 25% test, random_state=7, stratified on y
X_tr, X_te, y_tr, y_te = None, None, None, None


In [ ]:
assert X_tr is not None and X_te is not None, "fill in the split above"
assert len(X_te) == round(len(X) * 0.25), f"test set should have {round(len(X) * 0.25)} rows, it has {len(X_te)}"
assert len(X_tr) + len(X_te) == len(X), "train and test together should contain every row"
assert set(X_tr.index).isdisjoint(X_te.index), "a row appears in both train and test"
assert abs(y_tr.mean() - y.mean()) < 0.02 and abs(y_te.mean() - y.mean()) < 0.03, \
    "the share of at-risk students should be the same in both: add stratify=y"
expected = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)[1]
assert list(X_te.index) == list(expected.index), "use test_size=0.25 and random_state=7"
print("Exercise 1.1 correct.")

---
# Part 2 · Encoding: categories into numbers

Models work with numbers. `dept` holds the words `"CE"`, `"CS"`,
`"BME"`, so we have to convert them.

**The tempting mistake:** `CE = 0`, `CS = 1`, `BME = 2`. That tells the
model `BME` is "bigger" than `CS`, and that `CS` sits exactly midway
between the other two. None of that is true.

**One-hot encoding** avoids it: one new column per category, holding
`1` for the matching row and `0` otherwise.

Scikit-learn objects follow one pattern, and you'll use it for the rest
of the lab:

| Method | What it does |
|---|---|
| `fit(data)` | **learns** something from the data (here: which categories exist) |
| `transform(data)` | **applies** what was learned |

> `fit` only ever sees the **training** data. `transform` is then used
> on both train and test.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

demo_encoder = OneHotEncoder(sparse_output=False)
demo_encoder.fit(X_train[["study_mode"]])

print("categories learned from train:", list(demo_encoder.categories_[0]))
print()
encoded = demo_encoder.transform(X_test[["study_mode"]])
print(pd.DataFrame(encoded, columns=demo_encoder.get_feature_names_out()).head(4))

### What if a new category appears later?

Real systems meet data the training set never contained. Suppose a
student arrives from a department `"IS"` that wasn't in the data. The
default encoder refuses:

In [ ]:
new_students = pd.DataFrame({"dept": ["CE", "IS"]})     # IS was never seen in training

strict = OneHotEncoder(sparse_output=False).fit(X_train[["dept"]])
try:
    strict.transform(new_students)
except ValueError as e:
    print("default encoder:", str(e)[:70], "...")

forgiving = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(X_train[["dept"]])
print()
print("columns:", list(forgiving.get_feature_names_out()))
print(forgiving.transform(new_students))

With `handle_unknown="ignore"` the unknown department becomes a row of
all zeros: "none of the departments I know". Your system keeps running
instead of crashing.

> One-hot suits categories with **no order** (department). For
> categories with a real order (`low < medium < high`) scikit-learn has
> `OrdinalEncoder`, where you state the order yourself.

### Exercise 2.1
Build `encoder` as a one-hot encoder that ignores unknown categories and
returns a plain array. **Fit it on `X_train[["dept"]]` only.** Then
produce:

- `dept_train`: the encoded `X_train[["dept"]]`
- `dept_new`: the encoded `new_students`

In [ ]:
# TODO: encoder = a OneHotEncoder (handle_unknown="ignore", sparse_output=False), fit on X_train[["dept"]]
encoder = None

dept_train = None      # TODO: encode X_train[["dept"]]
dept_new = None        # TODO: encode new_students


In [ ]:
assert encoder is not None and dept_train is not None and dept_new is not None, "fill in all three"
assert list(encoder.categories_[0]) == sorted(X_train["dept"].unique()), \
    "the encoder should learn its categories from X_train only"
n_cats = X_train["dept"].nunique()
assert dept_train.shape == (len(X_train), n_cats), f"dept_train should have shape ({len(X_train)}, {n_cats})"
assert dept_new.shape == (2, n_cats), f"dept_new should have shape (2, {n_cats})"
assert dept_new[1].sum() == 0, "the unseen department IS should become all zeros: use handle_unknown='ignore'"
assert dept_new[0].sum() == 1, "CE should have exactly one 1"
print("Exercise 2.1 correct.")

---
# Part 3 · Scaling and missing values

Look at the numeric columns: `attendance` is a percentage in the tens
(up to 100), while `phone_hours` is a single digit and `credit_load` sits
between 12 and 18. Many methods treat a bigger number as a bigger
signal, so a column measured in the tens can drown out one measured in
single digits, just because of the units.

**Scaling** puts columns on a comparable footing:

| Scaler | Result |
|---|---|
| `StandardScaler` | subtract the mean, divide by the standard deviation: centred on 0 |
| `MinMaxScaler` | squeeze values into the range 0 to 1 |

**Missing values.** In Lab 5, filling a gap was a decision you recorded.
Here the decision is mechanical: `SimpleImputer(strategy="median")`
fills each gap with that column's median. The median is the value
**learned from the data**, so it follows the same rule:

> Learn the median from the training set only.

Here is one column, step by step:

In [ ]:
from sklearn.preprocessing import StandardScaler

hours_scaler = StandardScaler()
hours_scaler.fit(X_train[["study_hours"]])

print("learned from train: mean =", round(float(hours_scaler.mean_[0]), 2),
      "| std =", round(float(hours_scaler.scale_[0]), 2))

train_hours = hours_scaler.transform(X_train[["study_hours"]])
test_hours = hours_scaler.transform(X_test[["study_hours"]])

print()
print("scaled train: mean =", round(float(train_hours.mean()), 3), "| std =", round(float(train_hours.std()), 3))
print("scaled test:  mean =", round(float(test_hours.mean()), 3), "| std =", round(float(test_hours.std()), 3))

The scaled **train** column has mean 0 and standard deviation 1,
exactly. The scaled **test** column does not: it is close, but not exact.

That is **correct**, not a bug. The test set was scaled using the
training set's numbers, just as it would be for a brand-new student in
real use. If the test mean were exactly 0, it would mean you had scaled
the test set with its own statistics.

### Exercise 3.1
Prepare all four numeric columns in `numeric`:

1. `imputer`: a median `SimpleImputer`, fit on `X_train[numeric]`
2. `scaler`: a `StandardScaler`, fit on the **imputed** training data
3. `train_scaled` and `test_scaled`: both sets imputed, then scaled

Everything is learned from training data only.

In [ ]:
from sklearn.impute import SimpleImputer

numeric = ["study_hours", "attendance", "phone_hours", "credit_load"]

# TODO: imputer (strategy="median") fit on X_train[numeric]
imputer = None

# TODO: scaler (StandardScaler) fit on the imputed training data
scaler = None

train_scaled = None     # TODO: impute then scale X_train[numeric]
test_scaled = None      # TODO: impute then scale X_test[numeric]


In [ ]:
assert imputer is not None and scaler is not None, "fill in imputer and scaler"
assert train_scaled is not None and test_scaled is not None, "fill in train_scaled and test_scaled"
assert not np.isnan(train_scaled).any() and not np.isnan(test_scaled).any(), "no missing values should remain"
assert np.allclose(imputer.statistics_, X_train[numeric].median()), \
    "the imputer should learn its medians from X_train only"
assert scaler.n_samples_seen_ == len(X_train), \
    f"the scaler saw {int(scaler.n_samples_seen_)} rows but the training set has {len(X_train)}: fit it on training data only"
assert np.allclose(train_scaled.mean(axis=0), 0, atol=1e-9), "train columns should be centred on 0"
assert not np.allclose(test_scaled.mean(axis=0), 0, atol=1e-6), \
    "test means are exactly 0: you scaled the test set with its own statistics"
print("Exercise 3.1 correct.")

---
# Part 4 · Pipelines: one object, no mistakes

In Part 3 you ran three steps by hand for each set, in the right order,
fitting on the right data. Easy to get wrong, and the mistake is
silent.

A **`Pipeline`** chains steps into one object. Calling `fit` runs every
step on the training data; calling `transform` applies them all to
anything else.

A **`ColumnTransformer`** applies *different* steps to *different*
columns, then joins the results side by side.

Numeric columns get the Part 3 steps in one object:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

numeric_out = numeric_steps.fit_transform(X_train[numeric])
print("train:", numeric_out.shape, "| mean of first column:", round(float(numeric_out[:, 0].mean()), 3))
print("test: ", numeric_steps.transform(X_test[numeric]).shape)

Now the `ColumnTransformer` syntax, on a small example: scale one
column and encode another.

Each entry is `(name, what to do, which columns)`:

In [ ]:
small = ColumnTransformer([
    ("hours", StandardScaler(), ["study_hours"]),
    ("dept", OneHotEncoder(sparse_output=False), ["dept"]),
])

small_out = small.fit_transform(X_train)
print("columns produced:", list(small.get_feature_names_out()))
print("shape:", small_out.shape)

Two rules, and they are the whole point:

- **`fit_transform` on the training data**: learn, then apply.
- **`transform` only on the test data**: apply, never learn.

### Exercise 4.1
Build `preprocess`, a `ColumnTransformer` that applies:

- to the four `numeric` columns: median-impute then standard-scale
- to the `categorical` columns: one-hot encoding that ignores unknown
  categories (`sparse_output=False`)

Put the numeric part **first**. Then compute `X_train_ready`
(`fit_transform` on `X_train`) and `X_test_ready` (`transform` on
`X_test`).

In [ ]:
numeric = ["study_hours", "attendance", "phone_hours", "credit_load"]
categorical = ["dept", "study_mode"]

# TODO: preprocess = ColumnTransformer([...])  numeric first, then categorical
preprocess = None

X_train_ready = None     # TODO: fit_transform on X_train
X_test_ready = None      # TODO: transform on X_test


In [ ]:
assert preprocess is not None and X_train_ready is not None and X_test_ready is not None, "fill in all three"
n_expected = len(numeric) + sum(X_train[c].nunique() for c in categorical)
assert X_train_ready.shape == (len(X_train), n_expected), \
    f"train should have shape ({len(X_train)}, {n_expected}), got {X_train_ready.shape}"
assert X_test_ready.shape == (len(X_test), n_expected), \
    f"test should have shape ({len(X_test)}, {n_expected}), got {X_test_ready.shape}"
assert not np.isnan(X_train_ready).any() and not np.isnan(X_test_ready).any(), "no NaN should remain"
assert np.allclose(X_train_ready[:, :4].mean(axis=0), 0, atol=1e-9), \
    "the four numeric columns should come first and be centred on 0"
assert not np.allclose(X_test_ready[:, :4].mean(axis=0), 0, atol=1e-6), \
    "use transform (not fit_transform) on the test set"
print("Exercise 4.1 correct.")

---
# Part 5 · Detect leakage, and validate

**Data leakage** means information your system won't have in real use
has slipped into training. It comes in two kinds.

### Kind 1: leaking through preprocessing

This is what Part 1 warned about: fitting an imputer, scaler or encoder
on data that includes the test rows. There's a simple way to catch it.
A fitted scaler remembers how many rows it saw:

In [ ]:
all_numeric = X[numeric].fillna(X[numeric].median())

leaky_scaler = StandardScaler().fit(all_numeric)

print("rows the leaky scaler saw:", int(leaky_scaler.n_samples_seen_))
print("rows in the training set: ", len(X_train))
print()
print("mean it learned for study_hours:", round(float(leaky_scaler.mean_[0]), 3))
print("mean from train only:           ", round(float(X_train["study_hours"].mean()), 3))

The leaky scaler saw 200 rows when the training set has 160. The
extra 40 are the test set. **If an object learned from more rows than
the training set contains, it leaked.**

### Exercise 5.1
Five code snippets are below. `X` is the whole dataset, and `X_tr` and
`X_te` are train and test. Which ones **leak**? Store their letters in
`leaky_snippets`, for example `["A", "C"]`.

**A**
```python
X_tr, X_te = train_test_split(X, test_size=0.2)
scaler.fit(X_tr)
X_tr, X_te = scaler.transform(X_tr), scaler.transform(X_te)
```

**B**
```python
X = scaler.fit_transform(X)
X_tr, X_te = train_test_split(X, test_size=0.2)
```

**C**
```python
imputer.fit(X)
X_tr, X_te = train_test_split(imputer.transform(X), test_size=0.2)
```

**D**
```python
encoder.fit(X)
X_tr, X_te = train_test_split(X, test_size=0.2)
```

**E**
```python
X_tr, X_te = train_test_split(X, test_size=0.2)
pipe.fit(X_tr)
X_te = pipe.transform(X_te)
```

In [ ]:
leaky_snippets = []


In [ ]:
assert leaky_snippets, "fill in leaky_snippets"
assert sorted(leaky_snippets) == ["B", "C", "D"], \
    "for each snippet ask: is anything fitted BEFORE the split? Fitting on X (the whole dataset) before splitting is a leak."
print("Exercise 5.1 correct.")
print("B, C and D all fit on the whole dataset before splitting. A and E split first.")

### Kind 2: leaking through a column

Sometimes a **column itself** contains the answer. Recall Part 1: we
dropped `gpa`. Here's why. The target `at_risk` was defined as
"`gpa` below 2.5":

In [ ]:
full = df.drop(columns="student_id")
strength = full.corr(numeric_only=True)["at_risk"].drop("at_risk").round(2)
print(strength.sort_values())

`gpa` has the strongest relationship with `at_risk` of any column. That
is not a good feature; it is the answer in disguise. We only know a
student's final GPA after the thing we're trying to predict has already
happened. A model trained with it would look brilliant and be useless on
a new student.

> A correlation heatmap (Lab 6) can hint at this, but it won't always
> catch it. The real test is a question: **"Would I know this value at
> the moment I need to make the prediction?"** If not, drop it.

### Exercise 5.2
Preprocessing output should be checked, not trusted. Write
`find_problems(train_ready, test_ready)` for two NumPy arrays. It
returns a **list of problem descriptions** (strings), and an empty list
if everything is fine. Check for:

1. train and test having a **different number of columns**
2. any **NaN** in either array
3. any **infinite** value in either array

In [ ]:
def find_problems(train_ready, test_ready):
    """Return a list of problems found. An empty list means the arrays look fine."""
    problems = []
    # TODO: check 1 - different number of columns
    # TODO: check 2 - any NaN in either array
    # TODO: check 3 - any infinite value in either array
    return problems


In [ ]:
good = np.ones((5, 3))
assert find_problems(good, good) == [], "two clean arrays should give an empty list"
assert find_problems(np.ones((5, 3)), np.ones((2, 4))), "different column counts should be reported"
assert find_problems(np.array([[1.0, np.nan]]), np.ones((1, 2))), "a NaN should be reported"
assert find_problems(np.ones((1, 2)), np.array([[1.0, np.inf]])), "an infinite value should be reported"
print("Exercise 5.2 correct.")

if X_train_ready is not None:
    print("your Exercise 4.1 output:", find_problems(X_train_ready, X_test_ready) or "no problems found")

---
# AI checkpoint

**Pattern: check where it split.**

### Step 1
Give an AI assistant this exact prompt, unmodified:

> "Write Python code using scikit-learn to preprocess a dataset for
> machine learning: scale the numeric columns, one-hot encode the
> categorical columns, fill missing values, and split into train and
> test. The data is in data/students_prep.csv with columns student_id,
> dept, study_mode, study_hours, attendance, phone_hours, credit_load,
> gpa, at_risk (the target)."

### Step 2
Paste its code below and run it.

In [ ]:
# Paste the AI assistant's code here and run it



### Step 3: answer these in your own words

**Q1. Order.** Find the line that splits the data. Which preprocessing
lines come **before** it? (Part 1.)

_[your answer]_

**Q2. Leaks.** Does anything get fitted on the whole dataset, such as a
scaler or the values used to fill gaps? How would you prove it? (Part 5.)

_[your answer]_

**Q3. Columns.** Look at the columns it kept as features. Is `gpa` among
them? `student_id`? Would you know those values at prediction time?
(Part 5.)

_[your answer]_

**Q4. The prompt.** Rewrite the prompt so the code you get back would be
correct.

_[your answer]_

> **The point:** the code runs, prints sensible shapes, and looks
> professional. Every one of those problems is invisible unless you go
> looking for it.

---
# Going further

| Part | Resource | Why it's worth your time |
|---|---|---|
| 1, 5 | [Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html) | scikit-learn's own page on data leakage and how pipelines prevent it. |
| 2, 3 | [Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html) | Every encoder and scaler, with when to use each. |
| 4 | [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html) | `Pipeline` and `ColumnTransformer` in full. |
| Feature engineering | [Kaggle Learn: Feature Engineering](https://www.kaggle.com/learn/feature-engineering) | A short free course on the topic below. |

### Optional: feature engineering

**Feature engineering** means creating new columns from existing ones,
to make a useful pattern easier to see. Two common moves are a ratio and
a binned version of a number:

In [ ]:
fe = df[["study_hours", "credit_load", "attendance"]].copy()

fe["hours_per_credit"] = fe["study_hours"] / fe["credit_load"]
fe["attendance_band"] = pd.cut(fe["attendance"], bins=[0, 70, 85, 100], labels=["low", "medium", "high"])

fe.head()

One caution connects back to Part 5. A feature that uses only the
**same row** (a ratio) is safe to create before splitting. A feature
that uses numbers **from other rows** (a group average, a z-score)
learns from data, so it must be learned from the training set only,
like every other fitted step.

---
# Before you finish

- [ ] Every self-check cell prints "correct"
- [ ] `scikit-learn` installed **and** recorded in `requirements.txt`
- [ ] AI checkpoint answered in your own words
- [ ] **Restart Kernel and Run All**: the whole notebook runs with no errors
- [ ] Committed and pushed

### The habits worth keeping

1. Split first. Learn from the training set only.
2. `fit` on train, `transform` on both. Never `fit` on test.
3. A scaled test set that isn't exactly centred is correct, not broken.
4. Use a pipeline, so the order can't go wrong.
5. Ask of every column: would I know this when I make the prediction?
6. Check the output (shapes, NaN, columns) instead of trusting it.